# Lab 03 — Signature Authenticity Verification

> **GraphoLab** | Forensic Graphology Laboratory

**Technique:** Pre-trained CNN feature extractor + cosine similarity  
**Backbone:** ResNet-18 (ImageNet weights, via `torchvision`) — no training required  
**Task:** Compare two signatures and determine genuine vs. forged  
**Forensic use case:** Bank cheques, contracts, wills, identity documents

---

## How Signature Verification Works

The core idea behind AI-based signature verification is **metric learning**: instead of classifying a signature as genuine or forged directly, we teach a model to measure *how similar* two signatures are.

```
Reference Signature ──► [Feature Extractor] ──► embedding_ref ─┐
                                                                  ├──► cosine similarity ──► genuine / forged
Questioned Signature ──► [Feature Extractor] ──► embedding_q  ─┘
```

- **High similarity** → similar visual style → likely **genuine**
- **Low similarity** → different visual style → likely **forged**

### Why a pre-trained ResNet-18?

Training a Siamese network from scratch (e.g. SigNet) requires large labelled signature datasets and significant compute. For this demo we use a **ResNet-18 backbone pre-trained on ImageNet** as the feature extractor:

- The lower convolutional layers already detect edges, curves, and texture — exactly the visual primitives that distinguish signatures.
- No training required: weights download automatically (~45 MB).
- Produces meaningful similarity scores immediately.

> **Production note:** Domain-specific models (e.g. SigNet trained on CEDAR or SigComp) outperform this baseline on real forensic datasets. See [luizgh/sigver](https://github.com/luizgh/sigver) for pre-trained SigNet weights.


## GraphoLab Core — Quick Start

> The production implementation of signature verification is available in [`core/signature.py`](../core/signature.py).
> It uses **SigNet** (Hafemann et al. 2017) — a Siamese CNN trained on GPDS-300 — which produces
> 2048-dimensional L2-normalised embeddings and significantly outperforms the ResNet-18 baseline
> used in this notebook.
>
> Run the cell below to import it directly. The remaining cells implement the **ResNet-18 baseline**
> from scratch for educational purposes.

In [ ]:
# GraphoLab Core — production usage
# Run this cell to use the shared core module instead of the notebook implementation below.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from core.signature import sig_verify, get_signet, SIG_THRESHOLD
from PIL import Image
import numpy as np

# Example: verify two signatures
# ref  = np.array(Image.open("../data/samples/signature_genuine_01.png").convert("RGB"))
# query = np.array(Image.open("../data/samples/signature_forged_01.png").convert("RGB"))
# weights = pathlib.Path("../data/signet.pth")
# report, chart = sig_verify(ref, None, query, weights)
# print(report)
print(f"core.signature imported — SIG_THRESHOLD={SIG_THRESHOLD:.2f}, sig_verify() ready.")

## Setup


In [ ]:
# !pip install torch torchvision Pillow matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image, ImageOps, ImageDraw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} — device: {DEVICE}")

## Load the Feature Extractor

We use ResNet-18 with ImageNet pre-trained weights. The classification head is removed — we only keep the convolutional backbone, which outputs a 512-dimensional feature vector per image.

Weights are downloaded automatically on first run (~45 MB) and cached locally.


In [ ]:
class SignatureFeatureExtractor(nn.Module):
    """ResNet-18 backbone with classification head removed.
    Outputs a 512-dimensional L2-normalised feature vector per image.
    """

    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Drop the final fully-connected layer
        self.backbone = nn.Sequential(*list(base.children())[:-1])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)          # (B, 512, 1, 1)
        features = features.view(features.size(0), -1)  # (B, 512)
        return F.normalize(features, p=2, dim=1)         # L2-normalise


print("Loading ResNet-18 (ImageNet pre-trained weights)...")
extractor = SignatureFeatureExtractor().to(DEVICE).eval()
print(f"Feature extractor ready — output dimension: 512")

## Image Preprocessing

Signatures are converted to RGB, normalised to a white background, and resized to 224×224 (ResNet-18 input size).


In [ ]:
TRANSFORM = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet stats
                std=[0.229, 0.224, 0.225]),
])


def preprocess(img: Image.Image) -> torch.Tensor:
    """Prepare a PIL signature image for the feature extractor."""
    img = img.convert('RGB')
    # Normalise background: ensure white background (signatures are dark on white)
    gray = np.array(img.convert('L'))
    if gray.mean() < 128:
        img = ImageOps.invert(img)
    return TRANSFORM(img).unsqueeze(0).to(DEVICE)  # (1, 3, 224, 224)


def load_signature(path: str | Path) -> Image.Image:
    return Image.open(path).convert('RGB')

## Verification Function


In [ ]:
# Decision threshold on cosine similarity: above → genuine, below → forged
# Calibrate this on your own dataset; 0.70 is a reasonable starting point
THRESHOLD = 0.70


def verify_signatures(img_ref: Image.Image, img_query: Image.Image) -> dict:
    """Extract features from two signature images and compute similarity."""
    t_ref = preprocess(img_ref)
    t_query = preprocess(img_query)

    with torch.no_grad():
        emb_ref = extractor(t_ref)    # (1, 512)
        emb_query = extractor(t_query)  # (1, 512)

    cosine_sim = F.cosine_similarity(emb_ref, emb_query).item()  # [-1, 1]
    verdict = "GENUINE" if cosine_sim >= THRESHOLD else "FORGED"

    return {
        "verdict": verdict,
        "cosine_similarity": cosine_sim,
        "threshold": THRESHOLD,
    }


def show_verification(img_ref: Image.Image, img_query: Image.Image,
                      result: dict, label_query: str = "Questioned") -> None:
    """Display the two signatures side by side with the verdict."""
    fig = plt.figure(figsize=(12, 4))
    gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[2, 2, 1.5])

    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_ref, cmap='gray')
    ax1.set_title("Reference Signature", fontsize=11)
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(img_query, cmap='gray')
    ax2.set_title(label_query, fontsize=11)
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    color = 'green' if result['verdict'] == 'GENUINE' else 'red'

    # Similarity bar
    sim = result['cosine_similarity']
    verdict_text = (
        f"Verdict: {result['verdict']}\n\n"
        f"Cosine similarity:\n{sim:.4f}\n\n"
        f"Threshold: {result['threshold']}"
    )
    ax3.text(0.1, 0.55, verdict_text, fontsize=11, va='center',
             color=color, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    # Mini similarity bar chart
    bar_ax = fig.add_axes([0.78, 0.18, 0.15, 0.25])
    bar_ax.barh([0], [sim], color=color, alpha=0.7)
    bar_ax.axvline(THRESHOLD, color='black', linestyle='--', linewidth=1)
    bar_ax.set_xlim(0, 1)
    bar_ax.set_yticks([])
    bar_ax.set_xlabel('sim', fontsize=8)
    bar_ax.set_title('score', fontsize=8)

    plt.suptitle("Signature Verification — ResNet-18 features",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Demo 1 — Load Real Signature Images

Place your images in `data/samples/`:
- `signature_genuine_01.png` — reference (known genuine)
- `signature_genuine_02.png` — second genuine sample (same person)
- `signature_forged_01.png` — questioned / forged signature


In [ ]:
samples_dir = Path("../data/samples")

ref_path     = samples_dir / "signature_genuine_01.png"
genuine_path = samples_dir / "signature_genuine_02.png"
forged_path  = samples_dir / "signature_forged_01.png"

all_real = ref_path.exists() and genuine_path.exists() and forged_path.exists()

if all_real:
    sig_ref     = load_signature(ref_path)
    sig_genuine = load_signature(genuine_path)
    sig_forged  = load_signature(forged_path)
    print("Real signature images loaded.")
else:
    print("Real images not found — generating synthetic placeholders.")
    print(f"Missing files:")
    for p in [ref_path, genuine_path, forged_path]:
        if not p.exists():
            print(f"  {p}")
    print("\nAdd real signature PNGs to data/samples/ and re-run for meaningful results.")

    def _synth_sig(label: str, slant: int = 0, noise: float = 0.0,
                   size=(280, 120)) -> Image.Image:
        img = Image.new('RGB', size, 'white')
        d = ImageDraw.Draw(img)
        x = 20
        rng = np.random.RandomState(abs(hash(label)) % 2**31)
        for _ in range(rng.randint(6, 10)):
            h = rng.randint(30, 55)
            dx = int(h * np.tan(np.radians(slant)))
            d.line([(x + dx, 70 - h), (x, 70)], fill='black', width=2)
            d.arc([(x, 50), (x + 25, 75)], 0, 180, fill='black', width=2)
            x += rng.randint(28, 40)
            if x > size[0] - 20:
                break
        d.line([(20, 85), (x, 85)], fill='black', width=1)
        if noise > 0:
            arr = np.array(img, dtype=np.float32)
            arr += np.random.normal(0, noise * 255, arr.shape)
            img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
        return img

    sig_ref     = _synth_sig("genuine",  slant=10)
    sig_genuine = _synth_sig("genuine",  slant=10, noise=0.02)
    sig_forged  = _synth_sig("forged",   slant=-5, noise=0.03)

## Demo 2 — Reference vs. Genuine Copy


In [ ]:
result_genuine = verify_signatures(sig_ref, sig_genuine)

print("Reference vs. Genuine:")
print(f"  Cosine similarity : {result_genuine['cosine_similarity']:.4f}")
print(f"  Threshold         : {result_genuine['threshold']}")
print(f"  Verdict           : {result_genuine['verdict']}")

show_verification(sig_ref, sig_genuine, result_genuine,
                  label_query="Genuine Copy")

## Demo 3 — Reference vs. Forged


In [ ]:
result_forged = verify_signatures(sig_ref, sig_forged)

print("Reference vs. Forged:")
print(f"  Cosine similarity : {result_forged['cosine_similarity']:.4f}")
print(f"  Threshold         : {result_forged['threshold']}")
print(f"  Verdict           : {result_forged['verdict']}")

show_verification(sig_ref, sig_forged, result_forged,
                  label_query="Questioned / Forged")

## Demo 4 — Use Your Own Images


In [ ]:
# ─── Change these paths to your own signature images ──────────────────────────
MY_REFERENCE = "../data/samples/signature_genuine_01.png"
MY_QUESTIONED = "../data/samples/signature_forged_01.png"
# ──────────────────────────────────────────────────────────────────────────────

p_ref = Path(MY_REFERENCE)
p_q   = Path(MY_QUESTIONED)

if p_ref.exists() and p_q.exists():
    my_ref = load_signature(p_ref)
    my_q   = load_signature(p_q)
    my_result = verify_signatures(my_ref, my_q)
    print(f"Similarity: {my_result['cosine_similarity']:.4f}  →  {my_result['verdict']}")
    show_verification(my_ref, my_q, my_result, label_query=p_q.name)
else:
    print("Files not found. Update the paths above and re-run.")

## Similarity Distribution Across Multiple Pairs

When multiple genuine and forged samples are available, plotting the similarity distribution helps set a principled threshold.


In [ ]:
genuine_files = sorted(samples_dir.glob("signature_genuine_*.png"))
forged_files  = sorted(samples_dir.glob("signature_forged_*.png"))

if len(genuine_files) >= 2 and len(forged_files) >= 1:
    ref_img = load_signature(genuine_files[0])

    genuine_scores = [
        verify_signatures(ref_img, load_signature(p))["cosine_similarity"]
        for p in genuine_files[1:]
    ]
    forged_scores = [
        verify_signatures(ref_img, load_signature(p))["cosine_similarity"]
        for p in forged_files
    ]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(range(len(genuine_scores)), genuine_scores,
               color='green', s=80, zorder=5, label='Genuine')
    ax.scatter([len(genuine_scores) + i for i in range(len(forged_scores))],
               forged_scores, color='red', s=80, zorder=5, label='Forged/Different')
    ax.axhline(THRESHOLD, color='black', linestyle='--', label=f'Threshold ({THRESHOLD})')
    ax.set_ylabel("Cosine Similarity")
    ax.set_ylim(0, 1)
    ax.set_title("Similarity Scores — All Pairs vs. Reference", fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Add multiple genuine_*.png and forged_*.png files to data/samples/ to see the distribution plot.")

## Forensic Notes

- **Threshold calibration:** The default threshold of 0.70 is a starting point. Calibrate it on a representative dataset of genuine and forged pairs for your specific use case.
- **Multiple reference samples:** Always compare against several genuine reference signatures (minimum 5–10), not just one.
- **Model limitations:** ResNet-18 ImageNet features are a general-purpose baseline. For production forensic use, domain-specific SigNet weights (trained on CEDAR or SigComp'11) provide significantly better performance — see [luizgh/sigver](https://github.com/luizgh/sigver).
- **Scan quality:** Use high-resolution scans (≥300 DPI) with consistent lighting and background.
- **AI is a screening tool:** A high similarity score supports — but does not prove — authenticity. Skilled forgeries may score high; unusual genuine signatures may score low.

---

**Next lab →** [04 — Signature Detection in Documents (Conditional DETR)](04_signature_detection_detr.ipynb)